# Packages utiles et configuration du répertoire de travail

Création d'un répertoire de travail dans lequel seront stockés les résultats, les jeux de poids, ....

In [ ]:
# Des packages classiques !
import numpy as np # Classique, utile pour manipuler des tableaux, des matrices
from matplotlib import pyplot as plt # Permet de générer des figures
import random # Permet de générer des valeurs aléatoires (entier entre 0 et ...)
import itertools # Permet d'agir de manière itérative sur des données structurées

# Apprentissage automatique
import tensorflow as tf   # Librairie de référence dans la programmation d'IA

import keras # Librairie de référence dans la programmation de réseau de neurones
from keras.models import Sequential # Permet d'arranger les différentes couches du réseau
from keras.models import Model  # Permet de construire un modèle de réseau de neurones
from keras.layers import Dense # Pour ajout d'une couche dense
from keras.layers import Flatten # Pour ajout d'une couche de flatten
from keras.layers import Dropout # Pour ajout d'une couche de dropout (utilisée pour la régularisation)
from keras.layers import Activation # Importe les différences fonction d'activation
from keras.layers import Conv2D # Pour ajout d'une couche de convolution 2D
from keras.layers import MaxPooling2D #Pour ajout d'une couche de MaxPooling
from tensorflow.keras.utils import to_categorical # A vous de trouver !

# Outil d'analyse des performances d'un modèle de classification supervisée
# Permet d'obtenir la matrice de confusion du modèle en croisant les labels et les prédictions
from sklearn.metrics import confusion_matrix

import sklearn.neighbors as knn  # Algorithme de classification supervisée (k plus proches voisins)
import sklearn.cluster as clstr  # Bibliothèque pour la classification non supervisée

import os # Utile pour naviguer à travers les différents dossiers
from pathlib import Path # Facilite la manipulation des chemins vers des fichiers
import glob # librairie spécialisée dans la recherche de chemin

############################################################################
# renseignez le chemin vers le dossier de travail
# dans lequel seront déposés les datasets et les résultats : jeux de poids, figures...
# Il reste juste à choisir le nom que vous voulez pour le working_dir
# (i.e. renommer 2025-2026-S5-Projet-MNIST-RGB )
############################################################################

working_dir="2025-2026-S5-Projet-MNIST-RGB"
if not os.path.exists(working_dir):  #Si le repertoire n'existe pas, celui-ci est créé
    os.mkdir(working_dir)
print("Repertoire de travail Jupyter : ", working_dir)

## Des fonctions utiles fournies :

Dans la cellule suivante, différentes fonctions sont définies. Le tableau ci-dessous en donne un récapitulatif

| Nom de la fonction | Paramètre(s) | Rôle |
|---|---|---|
| summarize_diagnostics | history, directory, filename | tracé de l'évolution de la loss et de l'accuracy au cours des époques d'entraînement (à la fois sur les données de train et de test). Permet de voir que l'entrainement s'est bien passé et notamment que le réseau ne fait pas d'overfitting |
| confusion_matrix | Ylabels, Ypredictions | tracé personnalisé d'une matrice de confusion = outil d'analyse des prédictions pour un problème de classification |
| occurencesclasses | label, numclasses | calcul du nombre d'occurences de chaque classe dans un dataset. Permet d'analyser la composition d'un dataset, en particulier vérifier que la répartition entre les classes est suffisamment équilibrée |
| ShowBadPredictions | X, predictions, labels | visualisation de l'ensemble des mauvaises prédictions. Affiche l'ensemble des images mal prédites avec pour chacune en titre la prédiction P et le vrai label T (T pour True). |

----
Pour rappel, la matrice de confusion est un outil agréable de représentation et d'analyse des résultats pour un problème de classification

La diagonale d'un matrice de confusion correspond donc aux bonnes prédictions, et elle permet en particulier de voir si des confusions se produisent plus spécifiquement entre certaines classes. En recherchant sur internet (par exemple "confusion matrix Keras", "confusion matrix plot"), vous trouverez de nombreux sites sur le sujet, qui vous présentent cet outils et vous montre comment tracer une matrice de confusion et personnaliser ce tracé à vos souhaits. Par exemples :

- https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics (section 3.3.2.6)
- https://medium.com/@dtuk81/confusion-matrix-visualization-fc31e3f30fea
- https://www.kaggle.com/grfiv4/plot-a-confusion-matrix

In [ ]:
def summarize_diagnostics(history,
                          directory,
                          filename):
    """
    pour visualiser l'entrainement du réseau,
    nous avons défini une procédure qui crée un graphique représentant l'évolution :
      - Des valeurs de la fonction de perte (loss)
      - De la précision de classification (accuracy) pour les données d'entraînement et de test

    Arguments
    ---------
    history:          historique de l'entraînement

    directory:        répertoire de sauvegarde des images

    filename:         Nom attribuée à l'image dans le dossier "directory"

    Usage
    -----
    summarize_diagnostics(history,
                          directory,
                          filename)
    """
    # plot loss
    plt.subplot(211)
    plt.title('Cross Entropy Loss')
    plt.plot(history.history['loss'], color='blue', label='train')
    plt.plot(history.history['val_loss'], color='orange', label='test')
    plt.legend()
    # plot accuracy
    plt.subplot(212)
    plt.title('Classification Accuracy')
    plt.plot(history.history['accuracy'], color='blue', label='train')
    plt.plot(history.history['val_accuracy'], color='orange', label='test')
    plt.legend()
    # save plot to file
    plt.savefig("{}/{}_plot.png".format(directory,filename))
    plt.show()
    plt.close()


def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=False):
    """
    given a sklearn confusion matrix (cm), make a nice plot

    Arguments
    ---------
    cm:           confusion matrix from sklearn.metrics.confusion_matrix

    target_names: given classification classes such as [0, 1, 2]
                  the class names, for example: ['high', 'medium', 'low']

    title:        the text to display at the top of the matrix

    cmap:         the gradient of the values displayed from matplotlib.pyplot.cm
                  see http://matplotlib.org/examples/color/colormaps_reference.html
                  plt.get_cmap('jet') or plt.cm.Blues

    normalize:    If False, plot the raw numbers
                  If True, plot the proportions

    Usage
    -----
    plot_confusion_matrix(cm           = cm,                  # confusion matrix created by
                                                              # sklearn.metrics.confusion_matrix
                          normalize    = True,                # show proportions
                          target_names = y_labels_vals,       # list of names of the classes
                          title        = best_estimator_name) # title of graph

    Citiation
    ---------
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

    """

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

def OccurencesClasses(label,numclasses):
  classes_list=np.arange(0,numclasses) # classes={0,1,...,N-1} pour un problème à N classes
  occurences=np.zeros(len(classes_list),int)
  for i in range(len(label)):
      classe=int(label[i])
      occurences[classe]=occurences[classe]+1
  return(occurences)

def ShowBadPredictions(X,predictions,labels):
    Lerreur=[]
    for i in range(len(labels)):
        if  predictions[i] != labels[i] :
            Lerreur.append(i)
    print('Nombre de prédictions erronées=',len(Lerreur),'')
    print('accuracy=', (len(labels)-len(Lerreur))/len(labels)*100)
    n=int(len(Lerreur)/10)+1
    plt.figure(figsize=(10,1.5*n))
    for i in range(len(Lerreur)):
        plt.subplot(n,10,i+1)
        ind=Lerreur[i]
        titre='P='+str(predictions[ind])+',T='+str(labels[ind])
        plt.title(titre,fontsize=12)
        plt.imshow(X[ind],cmap="Greys")
        plt.axis("off")
    plt.show()

# Chargement du dataset MNIST, puis création d'une variante "MNIST RGB"

La base de données MNIST (pour Modified ou Mixed National Institute of Standards and Technology) est une des plus célèbres base de données labelisée d'images. Elle aborde le problème de reconnaissance automatique de nombres en classification par apprentissage automatique (Machine Learning). Cette base de données d'images en niveaux de gris, de chiffres écrits "à la main", regroupe 60000 images d'apprentissage et 10000 images de test. La taille de chaque image est de 28 * 28 = 784 pixels.

Remarque : Sur ce même format (images 28x28 en niveau de gris), de nombreuses variantes de cette célèbre base de données MNIST ont été créées, dont Fashion MNIST (base de donnée d'articles Zalando). Fashion MNIST comporte, comme pour la base de données MNIST, 60000 images d'apprentissage et 10000 images de test, en niveaux de gris, de taille 28 * 28 = 784 pixels, reparties en 10 classes différentes : [ 0:"T-shirt/top", 1:"Trouser", 2:"Pullover", 3:"Dress", 4:"Coat", 5:"Sandal", 6:"Shirt", 7:"Sneaker", 8:"Bag", 9:"Ankle boot" ]. Comme pour MNIST, les images de cette base de données Fashion MNIST peuvent etre directement importées depuis KERAS. cette base de données est plus "challenging" que la base de données MNIST.

In [ ]:
# Importation de MNIST (et de Fashion MNIST)

from keras.datasets import mnist, fashion_mnist

(Xtrain, Ytrain), (Xtest, Ytest) = mnist.load_data()
(XtrainFM, YtrainFM), (XtestFM, YtestFM) = fashion_mnist.load_data()

print('\n MNIST')
print('shape xtrain=',Xtrain.shape)
print('shape ytrain=',Ytrain.shape)
print('shape xtest=',Xtest.shape)
print('shape ytest=',Ytest.shape)

print('\n Fashion MNIST')
print('shape xtrain=',XtrainFM.shape)
print('shape ytrain=',YtrainFM.shape)
print('shape xtest=',XtestFM.shape)
print('shape ytest=',YtestFM.shape)

# Nombre d'occurrences dans chacune des 10 classes pour le dataset de Train et de Test
print('Train MNIST :', OccurencesClasses(Ytrain,10))
print('Test MNIST :', OccurencesClasses(Ytest,10))
print('Train Fashion MNIST :', OccurencesClasses(YtrainFM,10))
print('Test Fashion MNIST :', OccurencesClasses(YtestFM,10))
# Nous constatons que le dataset MNIST est assez bien équilibré
# et que Fashion MNIST est parfaitement équilibré

Affichage des 50 premières images d'entrainement avec les labels

In [ ]:
print('Des images labélisées  MNIST')
plt.figure(figsize=(7,5))
for i in range(50):
    plt.subplot(5,10,i+1)
    plt.title(Ytrain[i])
    plt.imshow(Xtrain[i],cmap="Greys")
    plt.axis("off")
plt.show()

print('\n')

print('Des images labélisées Fashion-MNIST')
plt.figure(figsize=(7,5))
for i in range(50):
    plt.subplot(5,10,i+1)
    plt.title(YtrainFM[i])
    plt.imshow(XtrainFM[i],cmap="Greys")
    plt.axis("off")
plt.show()

## Création d'une variante : "MNIST RGB"

Création d'une base de données MNIST RGB avec

Point des chiffres dans ['noir' 'blanc', 'rouge', 'vert', 'bleu', 'jaune', 'magenta', 'cyan'] i.e.

ListRGB=[[0.,0.,0.],[1.,1.,1.],[1.,0.,0.],[0.,1.,0.],[0.,0.,1.],[1.,1.,0],[1.,0,1.],[0,1.,1.]]

Fond uni de composantes RGB choisies aléatoirement selon une loi uniforme dans [0.2,0.8]. Pour chaque composante R,G,B : np.round(random.uniform(0.2,0.8),2) où 2 désigne le nombre de décimales

In [ ]:
# Liste de couleurs pour les chiffres
ListCouleurs=['noir','blanc','rouge','vert','bleu','jaune','magenta','cyan']
ListRGB=np.array([[0.,0.,0.],[1.,1.,1.],[1.,0.,0.],[0.,1.,0.],[0.,0.,1.],[1.,1.,0],[1.,0,1.],[0,1.,1.]], dtype=np.float32)
# Remarque : ajout d'un . derrière les 0 et 1 pour préciser que nous travaillons avec des réels

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def make_rgb(X, rng):
    """Fond uniforme dans [0.2, 0.8]^3 (arrondi à 2 décimales), chiffre de l'une des 8 couleurs."""
    n = X.shape[0]
    fond = np.round(rng.uniform(0.2, 0.8, size=(n, 3)), 2).astype(np.float32)
    ind = rng.integers(0, 8, size=n)
    chiffre = ListRGB[ind]
    masque = (X > 0)[..., None]                       # (n, 28, 28, 1)
    XRGB = np.where(masque, chiffre[:, None, None, :], fond[:, None, None, :])
    return XRGB.astype(np.float32), fond, ind


# Création reproductible du dataset MNIST RGB (graine fixée)
rng = np.random.default_rng(SEED)
XtrainRGB, fond_tr, ind_tr = make_rgb(Xtrain, rng)
XtestRGB, fond_te, ind_te = make_rgb(Xtest, rng)

# Dimension des données de Train et de Test
print("Shape Xtrain RGB:", np.shape(XtrainRGB), ", Shape XtestRGB :", np.shape(XtestRGB))

# Affichage des 50 premières images MNIST RGB de train avec leur label
plt.figure(2,figsize=(10,6))
for i in range(50):
    plt.subplot(5,10,i+1)
    plt.title(Ytrain[i])
    plt.imshow(XtrainRGB[i])
    plt.axis("off")
plt.show()

In [ ]:
# Affichage des 50 premières images MNIST RGB de test avec leur label

plt.figure(2,figsize=(10,6))
for i in range(50):
    plt.subplot(5,10,i+1)
    plt.title(Ytest[i])
    plt.imshow(XtestRGB[i])
    plt.axis("off")
plt.show()

# "Travail Commun" - Reseau de neurones convolutif pour classification supervisée des chiffres " MNIST RGB" (10 classes)

Nous devons "préparer" les données afin qu'elles soient utilisables par Keras.

In [ ]:
# Transformation des entrées en array (entrée = tenseur)
# Lée réseau (couche Conv2D de convolution prend obligatoirement en entrée un tenseur)
XtrainRGB=np.array(XtrainRGB);
XtestRGB=np.array(XtestRGB);

# Codage des sorties label => vecteur
#########################################################################
# Il sera demandé d'expliquer cette mise au format dans le rapport
# Attention ! : vous devez bien comprendre ce que font ces lignes de code
#              ... et comprendre pourquoi il faudra utiliser Ytest et non
#                  Ytest_nn pour la matrice de confusion !
#########################################################################
Ytrain_nn = to_categorical(Ytrain)
Ytest_nn = to_categorical(Ytest)  # Utile juste pour regarder l'évolution de la loss et de l'accuracy sur le test


print('shape sorties avant mise au format : Ytrain,Ytest=',np.shape(Ytrain),np.shape(Ytest))
print('shape entrées et sorties après mise au format sorties:')
print("Xtrain:"+ str(np.shape(XtrainRGB)) + " , Ytrain_nn:" + str(np.shape(Ytrain_nn)))
print("Xtest:"+ str(np.shape(XtestRGB))+ " , Ytest nn:" + str(np.shape(Ytest_nn)))

La couche de sortie doit contenir autant de neurones qu'il y a de classes, avec une fonction d'activation softmax

Pour la fonction de loss et la métrique, nous choisissons " : 'categorical_crossentropy' https://keras.io/api/losses/ et 'Adam' pour optimizer : https://keras.io/api/optimizers/

Nous prendrons un moment pour expliquer la sortie "softmax" et la métrique "accuracy"

In [ ]:
model = Sequential()
#######################################################
# A Completer : Architecture de votre réseau
# vous pouvez changer le nom Model si vous voulez
########################################################
model.add(Conv2D(16, (5,5), activation='relu', input_shape=(28,28,3)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(32, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(32, (3,3), activation='relu'))
model.add(Flatten())
model.add(Dense(100, activation='relu', name='couche_1'))
model.add(Dense(40, activation='relu', name = 'couche_2'))
model.add(Dense(10, activation='softmax'))


# compilation du modèle avec choix de l'optimizer, de la fonction de loss et de la métrique
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary() # permet d'afficher l'architecture du réseaux, les noms et dimensions des couches, le nombre de paramètres

# Analyse des résultats

In [ ]:
###########################################################
# ici il reste juste à choisir le nom que vous voulez
# pour le fichier filesave avant d'executer la cellule
# L'entrainement prend un peu de temps.
#################################################################
# Une fois l'entrainement concluant et le jeu de poids du réseau
# entrainé sauvegardé, passer la variable TrainReseau à False
# cela évitera un train inutile lors d'une prochaine compilation
# du notebook qui prendrai du temps, aboutirai à des résultats différents
#et en plus écraserai le jeu de poids précédemment sauvegardé.
#################################################################

TrainReseau=False
filesave="weights_cnn_10classes"

# If TrainReseau==True => train & save weights. Else load weights

if TrainReseau :
    # Entraînement du modèle
    history = model.fit(XtrainRGB,Ytrain_nn, epochs=15, batch_size=64,
                          validation_data=(XtestRGB,Ytest_nn), verbose=1)

    # Sauvegarde du jeu de poids du modèle entraîné
    print('Sauvegarde du jeu de poids'," {}/{}.weights.h5".format(working_dir,filesave))
    model.save_weights("{}/{}.weights.h5".format(working_dir,filesave))
    # Courbes d'évolution de la loss et de l'accuracy au cours des époques, pour les données de train et de test
    summarize_diagnostics(history,working_dir,filesave)

else :
    # Il sera possible à tout moment de procéder au chargement du jeu de poids
    # précédemment sauvegardé dans le dossier working_dir :
#     print('Chargement du jeu de poids'," {}/{}.weights.h5".format(working_dir,filesave))
    model.load_weights("{}/{}.weights.h5".format(working_dir,filesave))

# model.compile (optimizer ='adam', loss ='categorical_crossentropy', metrics=['accuracy'])


# Evaluation du modèle sur les données de test
_, acc = model.evaluate(XtestRGB,Ytest_nn)
print('Accuracy sur les données de test > %.3f' % (acc * 100.0))

In [ ]:
# Nombre d'occurrences dans chacune des 10 classes pour le dataset de Test
print('Test :', OccurencesClasses(Ytest, 10))

# Prédictions et matrice de confusion
y_pred = model.predict(XtestRGB)
Y_pred = np.argmax(y_pred, 1)

cm = confusion_matrix(Ytest, Y_pred)

Mytitle = 'matrice_de_confusion'
Names_Label = ['0','1','2','3','4','5','6','7','8','9']
Mycmap = 'Blues'

plot_confusion_matrix(cm, title=Mytitle, target_names=Names_Label, cmap=Mycmap, normalize=False)